## import 구문들

In [1]:
!pip install SimpleITK

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 MB 12.5 MB/s eta 0:00:00


In [2]:
!pip install pyradiomics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 20.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Discarding https://files.pythonhosted.org/packages/03/c1/20fc2c50ab1e3304da36d866042a1905a2b05a1431ece35448ab6b4578f2/pyradiomics-3.1.0.tar.gz (from https://pypi.org/simple/pyradiomics/): Requested pyradiomics from https://files.pythonhosted.org/packages/03/c1/20fc2c50ab1e3304da36d866042a1905a2b05a1431ece35448ab6b4578f2/pyradiomics-3.1.0.tar.gz has inconsistent version: expected '3.1.0', but metadata has '3.0.1a1'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 44.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.8/117.8 kB 17.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.7/526.7 kB 56.5 MB/s eta 0:

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
cd ./drive/MyDrive/Colab/

/content/drive/MyDrive/Colab


In [5]:
from __future__ import print_function

import SimpleITK as sitk
import numpy as np
import six
import cv2
import glob
import sys, os

from radiomics import firstorder, getTestCase, glcm, glrlm, glszm, imageoperations, shape, shape2D, featureextractor
import copy

%matplotlib inline
import matplotlib.pyplot as plt

import pandas as pd


## 데이터폴더 내에 존재하는 영상데이터에서 전체 feature 추출

In [6]:
DF = pd.DataFrame()

for folder in glob.glob("data/sample_dcm_roi/*/"): #ROI 하위 폴더에 존재하는 폴더 탐색
#     try:
    print(folder)
    pat = folder.split('/')[1]#탐색된 폴더 이름(환자 ID) 저장
    print(folder)
    for dira in glob.glob(folder+'*.png'):#폴더 내 png 파일 탐색
        label = os.path.basename(dira).split('_')[0]
        dcm_file = dira[:-4]+".dcm"
        print(dcm_file)

        #데이터 load 및 전처리
        image = sitk.ReadImage(dcm_file)
        image = sitk.GetArrayFromImage(image)
        image = np.resize(image, [512, 512])
        print(image.shape)
        image_roi = sitk.ReadImage(dira)
        image_roi = sitk.GetArrayFromImage(image_roi)
        image_roi = np.resize(image_roi, [512, 512])
        image_roi = np.where(image_roi > 0, 1, 0)


        print(image_roi.shape)

        mask_roi = sitk.GetImageFromArray(image_roi, isVector=False)
        image = sitk.GetImageFromArray(image, isVector=False)

        #Radiomics 설정 parameter 조정
        applyLog = True
        applyWavelet = True

        settings = {'binWidth': 25,
                    'interpolator': sitk.sitkBSpline,
                    'resampledPixelSpacing': None}

        interpolator = settings.get('interpolator')
        resampledPixelSpacing = settings.get('resampledPixelSpacing')

        #설정된 사항에 맞춰서 데이터 crop 및 resampling
        if interpolator is not None and resampledPixelSpacing is not None:
            image, mask = imageoperations.resampleImage(image, mask, **settings)

        bb, correctedMask = imageoperations.checkMask(image, mask_roi)
        if correctedMask is not None:
            mask = correctedMask

        image, mask = imageoperations.cropToTumorMask(image, mask_roi, bb)

        #추출 시작
        extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
        extractor.enableAllFeatures()
        results = extractor.execute(image, mask)

        #추출된 radiomics feature를 dataframe에 저장
        csv_RM = pd.DataFrame.from_dict(results, orient='index')
        csv_RM.columns = ['Value']
        csv_RM.reset_index(level=0, inplace=True)
        csv_RM = csv_RM.T
        csv_RM['label'] = ['label',label]
        csv_RM['pat'] = ['pat',pat]
        new_row = pd.DataFrame([csv_RM.iloc[1]])
        DF = pd.concat([DF, new_row], ignore_index=True)
        print(DF)

#만들어진 dataframe을 csv 파일로 저장
DF.columns = list(csv_RM.loc['index'].values)
DF.to_csv('result_All.csv', index=True, na_rep='NaN')

data/sample_dcm_roi/00000001/
data/sample_dcm_roi/00000001/
data/sample_dcm_roi/00000001/nan_65.dcm
(512, 512)


INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:

(512, 512)
        0       1      2      3        4  \
0  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

                                                   5                 6  \
0  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   

                                          7   8           9  ...  \
0  862ca842c88c5f15d4a8baec728115cc10e43d04  2D  (1.0, 1.0)  ...   

                  116                 117                  118  \
0  6.0900831784019225  0.7925625660909411  0.41826634206610896   

                  119                    120                121  \
0  0.6975755748122952  0.0017045110265764087  3955.882684774056   

                   122                 123  label             pat  
0  0.25459301545891005  1.5750679158673369    nan  sample_dcm_roi  

[1 rows x 126 columns]
data/sample_dcm_roi/00000001/honeycombing_72.dcm
(512, 512)


INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:

(512, 512)
        0       1      2      3        4  \
0  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

                                                   5                 6  \
0  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   
1  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   

                                          7   8           9  ...  \
0  862ca842c88c5f15d4a8baec728115cc10e43d04  2D  (1.0, 1.0)  ...   
1  83a869d5c560b8170f95a746a110765b65f0ed2c  2D  (1.0, 1.0)  ...   

                  116                 117                  118  \
0  6.0900831784019225  0.7925625660909411  0.41826634206610896   
1   5.838858765353258  0.8154897494305239   0.3091429730657595   

                   119                    120                121  \
0   0.6975755748122952  0.0017045110265764087  3955.882684774056   
1  0.10516591396061917   0.016192227862355878  3880.071563914968   

                   122  

...

(512, 512)
         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:

(512, 512)


INFO:radiomics.featureextractor:Computing ngtdm


         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
21  v3.0.1

## 원하는 class만 골라서 추출하기

In [7]:
DF = pd.DataFrame()

for folder in glob.glob("data/sample_dcm_roi/*/"): #ROI 하위 폴더에 존재하는 폴더 탐색
#     try:
    pat = folder.split('/')[1]#탐색된 폴더 이름(환자 ID) 저장
    print(folder)
    for dira in glob.glob(folder+'*.png'):#폴더 내 png 파일 탐색
        label = os.path.basename(dira).split('_')[0]
        dcm_file = dira[:-4]+".dcm"
        print(dcm_file)

        #데이터 load 및 전처리
        image = sitk.ReadImage(dcm_file)
        image = sitk.GetArrayFromImage(image)
        image = np.resize(image, [512, 512])
        print(image.shape)
        image_roi = sitk.ReadImage(dira)
        image_roi = sitk.GetArrayFromImage(image_roi)
        image_roi = np.resize(image_roi, [512, 512])
        image_roi = np.where(image_roi > 0, 1, 0)


        print(image_roi.shape)

        mask_roi = sitk.GetImageFromArray(image_roi, isVector=False)
        image = sitk.GetImageFromArray(image, isVector=False)

        #Radiomics 설정 parameter 조정
        applyLog = True
        applyWavelet = True

        settings = {'binWidth': 25,
                    'interpolator': sitk.sitkBSpline,
                    'resampledPixelSpacing': None}

        interpolator = settings.get('interpolator')
        resampledPixelSpacing = settings.get('resampledPixelSpacing')

        #설정된 사항에 맞춰서 데이터 crop 및 resampling
        if interpolator is not None and resampledPixelSpacing is not None:
            image, mask = imageoperations.resampleImage(image, mask, **settings)

        bb, correctedMask = imageoperations.checkMask(image, mask_roi)
        if correctedMask is not None:
            mask = correctedMask

        image, mask = imageoperations.cropToTumorMask(image, mask_roi, bb)

        #추출 시작
        extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
        ###########
        ###이부분 주목

        extractor.disableAllFeatures() #추출할 항목 전체 disable
        extractor.enableFeatureClassByName('firstorder') # 원하는 class의 추출을 할 feature들 선택

        ###### 여기까지
        ################
        results = extractor.execute(image, mask)

        #추출된 radiomics feature를 dataframe에 저장
        csv_RM = pd.DataFrame.from_dict(results, orient='index')
        csv_RM.columns = ['Value']
        csv_RM.reset_index(level=0, inplace=True)
        csv_RM = csv_RM.T
        csv_RM['label'] = ['label',label]
        csv_RM['pat'] = ['pat',pat]
        new_row = pd.DataFrame([csv_RM.iloc[1]])
        DF = pd.concat([DF, new_row], ignore_index=True)
        #DF = DF.append([list(csv_RM.iloc[1])])
        print(DF)

#만들어진 dataframe을 csv 파일로 저장
DF.columns = list(csv_RM.loc['index'].values)
DF.to_csv('result_firstorder.csv', index=True, na_rep='NaN')

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

data/sample_dcm_roi/00000001/
data/sample_dcm_roi/00000001/nan_65.dcm
(512, 512)
(512, 512)
        0       1      2      3        4  \
0  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

                                                   5                 6  \
0  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   

                                          7   8           9  ...       32  \
0  862ca842c88c5f15d4a8baec728115cc10e43d04  2D  (1.0, 1.0)  ...  -1024.0   

       33                 34                 35                   36  \
0  1259.0  140.8979673192179  565.3411429043671  0.47473084331130017   

             37                    38                 39  label  \
0  1813470589.0  0.031196601532520876  54022.60569875839    nan   

              pat  
0  sample_dcm_roi  

[1 rows x 42 columns]
data/sample_dcm_roi/00000001/honeycombing_72.dcm
(512, 512)


INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

(512, 512)
        0       1      2      3        4  \
0  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

                                                   5                 6  \
0  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   
1  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   

                                          7   8           9  ...       32  \
0  862ca842c88c5f15d4a8baec728115cc10e43d04  2D  (1.0, 1.0)  ...  -1024.0   
1  83a869d5c560b8170f95a746a110765b65f0ed2c  2D  (1.0, 1.0)  ...  -1009.0   

       33                  34                 35                   36  \
0  1259.0   140.8979673192179  565.3411429043671  0.47473084331130017   
1  1278.0  147.59917533136905  638.9382476284075    0.552908954270086   

             37                    38                 39         label  \
0  1813470589.0  0.031196601532520876  54022.60569875839           nan   
1   179218275.0  0.03234209037935

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

...

         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
21  v3.0.1

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

(512, 512)
(512, 512)
         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0 

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
21  v3.0.1

## 원하는 feature만 골라서 추출하기

In [8]:
DF = pd.DataFrame()

for folder in glob.glob("data/sample_dcm_roi/*/"): #ROI 하위 폴더에 존재하는 폴더 탐색
#     try:
    pat = folder.split('/')[1]#탐색된 폴더 이름(환자 ID) 저장
    print(folder)
    for dira in glob.glob(folder+'*.png'):#폴더 내 png 파일 탐색
        label = os.path.basename(dira).split('_')[0]
        dcm_file = dira[:-4]+".dcm"
        print(dcm_file)

        #데이터 load 및 전처리
        image = sitk.ReadImage(dcm_file)
        image = sitk.GetArrayFromImage(image)
        image = np.resize(image, [512, 512])
        print(image.shape)
        image_roi = sitk.ReadImage(dira)
        image_roi = sitk.GetArrayFromImage(image_roi)
        image_roi = np.resize(image_roi, [512, 512])
        image_roi = np.where(image_roi > 0, 1, 0)


        print(image_roi.shape)

        mask_roi = sitk.GetImageFromArray(image_roi, isVector=False)
        image = sitk.GetImageFromArray(image, isVector=False)

        #Radiomics 설정 parameter 조정
        applyLog = True
        applyWavelet = True

        settings = {'binWidth': 25,
                    'interpolator': sitk.sitkBSpline,
                    'resampledPixelSpacing': None}

        interpolator = settings.get('interpolator')
        resampledPixelSpacing = settings.get('resampledPixelSpacing')

        #설정된 사항에 맞춰서 데이터 crop 및 resampling
        if interpolator is not None and resampledPixelSpacing is not None:
            image, mask = imageoperations.resampleImage(image, mask, **settings)

        bb, correctedMask = imageoperations.checkMask(image, mask_roi)
        if correctedMask is not None:
            mask = correctedMask

        image, mask = imageoperations.cropToTumorMask(image, mask_roi, bb)

        #추출 시작
        extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
        ###########
        ###이부분 주목

        extractor.disableAllFeatures() #추출할 항목 전체 disable
        extractor.enableFeaturesByName(firstorder=['Kurtosis','Maximum'],
                                      glcm=['ClusterProminence','ClusterShade','Correlation'],
                                      gldm=['LargeDependenceLowGrayLevelEmphasis'],
                                      glszm=['LargeAreaHighGrayLevelEmphasis','LargeAreaLowGrayLevelEmphasis','SizeZoneNonUniformity','SmallAreaLowGrayLevelEmphasis','ZoneVariance'],
                                      ngtdm=['Busyness','Coarseness']) # 원하는 class의 추출을 할 feature들 선택
        ###### 여기까지
        ################
        results = extractor.execute(image, mask)

        #추출된 radiomics feature를 dataframe에 저장
        csv_RM = pd.DataFrame.from_dict(results, orient='index')
        csv_RM.columns = ['Value']
        csv_RM.reset_index(level=0, inplace=True)
        csv_RM = csv_RM.T
        csv_RM['label'] = ['label',label]
        csv_RM['pat'] = ['pat',pat]
        new_row = pd.DataFrame([csv_RM.iloc[1]])
        DF = pd.concat([DF, new_row], ignore_index=True)
        print(DF)

#만들어진 dataframe을 csv 파일로 저장
DF.columns = list(csv_RM.loc['index'].values)
DF.to_csv('result_byname.csv', index=True, na_rep='NaN')

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

data/sample_dcm_roi/00000001/
data/sample_dcm_roi/00000001/nan_65.dcm
(512, 512)
(512, 512)
        0       1      2      3        4  \
0  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

                                                   5                 6  \
0  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   

                                          7   8           9  ...  \
0  862ca842c88c5f15d4a8baec728115cc10e43d04  2D  (1.0, 1.0)  ...   

                     27                 28                   29  \
0  0.023833379874086787  964.0829441850123  0.01711917539656079   

                  30                    31                   32  \
0  3051.401601067378  0.005280588832362909  0.41826634206610896   

                   33                     34  label             pat  
0  0.6975755748122952  0.0017045110265764087    nan  sample_dcm_roi  

[1 rows x 37 columns]
data/sample_dcm_roi/00000001/honeycombing_72.dcm
(512, 512)


INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

(512, 512)
        0       1      2      3        4  \
0  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

                                                   5                 6  \
0  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   
1  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   

                                          7   8           9  ...  \
0  862ca842c88c5f15d4a8baec728115cc10e43d04  2D  (1.0, 1.0)  ...   
1  83a869d5c560b8170f95a746a110765b65f0ed2c  2D  (1.0, 1.0)  ...   

                     27                 28                   29  \
0  0.023833379874086787  964.0829441850123  0.01711917539656079   
1   0.03399871389112824   690.659217877095  0.02685381214722194   

                  30                    31                   32  \
0  3051.401601067378  0.005280588832362909  0.41826634206610896   
1  251.3240223463687  0.012228047316068797   0.3091429730657595   

                    33  

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

...

         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
21  v3.0.1

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

(512, 512)
(512, 512)
         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0 

INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Applying custom setting overrides: {'binWidth': 25, 'interpolator': 23, 'resampledPixelSpacing': None}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:C

(512, 512)
         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

## 만약 여러 class를 선택해서 추출하고 싶다면?

In [9]:
DF = pd.DataFrame()

for folder in glob.glob("data/sample_dcm_roi/*/"):
#     try:
    pat = folder.split('/')[1]
    print(folder)
    for dira in glob.glob(folder+'*.png'):
        label = os.path.basename(dira).split('_')[0]
        dcm_file = dira[:-4]+".dcm"

        image = sitk.ReadImage(dcm_file)
        image = sitk.GetArrayFromImage(image)
        image = np.resize(image, [512, 512])
        print(image.shape)
        image_roi = sitk.ReadImage(dira)
        image_roi = sitk.GetArrayFromImage(image_roi)
        image_roi = np.resize(image, [512, 512])
        image_roi = np.where(image > 0, 1, image_roi)

        print(image_roi.shape)

        mask = sitk.GetImageFromArray(image_roi, isVector=False)
        image = sitk.GetImageFromArray(image, isVector=False)

        applyLog = True
        applyWavelet = True

        settings = {'binWidth': 25,
                    'interpolator': sitk.sitkBSpline,
                    'resampledPixelSpacing': None}

        #firstorder 추출
        firstOrderFeatures = firstorder.RadiomicsFirstOrder(image, mask, **settings)
        firstOrderFeatures.enableAllFeatures()
        results = firstOrderFeatures.execute()

        results_first_order = {}
        for (key, val) in six.iteritems(results):
            results_first_order.update({key : val})

        csv_first_order = pd.DataFrame.from_dict(results_first_order, orient='index')
        csv_first_order.columns = ['Value']
        csv_first_order.reset_index(level=0, inplace=True)

        #GLCM 추출
        glcmFeatures = glcm.RadiomicsGLCM(image, mask, **settings)
        glcmFeatures.enableAllFeatures()

        results_GLCM = glcmFeatures.execute()

        csv_GLCM = pd.DataFrame.from_dict(results_GLCM, orient='index')
        csv_GLCM.columns = ['Value']
        csv_GLCM.reset_index(level=0, inplace=True)

        #GLRLM 추출
        glrlmFeatures = glrlm.RadiomicsGLRLM(image, mask, **settings)
        glrlmFeatures.enableAllFeatures()

        results_GLRLM = glrlmFeatures.execute()

        csv_GLRLM = pd.DataFrame.from_dict(results_GLRLM, orient='index')
        csv_GLRLM.columns = ['Value']
        csv_GLRLM.reset_index(level=0, inplace=True)

        #GLSZM 추출
        glszmFeatures = glszm.RadiomicsGLSZM(image, mask, **settings)
        glszmFeatures.enableAllFeatures()

        results_GLSZM = glszmFeatures.execute()

        csv_GLSZM = pd.DataFrame.from_dict(results_GLSZM, orient='index')
        csv_GLSZM.columns = ['Value']
        csv_GLSZM.reset_index(level=0, inplace=True)

        #추출 끝

        frames = [csv_first_order,csv_GLCM,csv_GLRLM,csv_GLSZM]
        All_results = pd.concat(frames)
        All_results = All_results.T
        All_results['label'] = ['label',label]
        All_results['pat'] = ['pat',pat]
        new_row = pd.DataFrame([csv_RM.iloc[1]])
        DF = pd.concat([DF, new_row], ignore_index=True)
        print(DF)

print(DF)
DF.columns = list(All_results.loc['index'].values)[:37]
DF.to_csv('result_bymulticlass.csv', index=True, na_rep='NaN')

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


data/sample_dcm_roi/00000001/
(512, 512)
(512, 512)


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


        0       1      2      3        4  \
0  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   

                                                   5                 6  \
0  {'minimumROIDimensions': 2, 'minimumROISize': ...  {'Original': {}}   

                                          7   8           9  ...  \
0  5cf5d30470f8125b2954925491258c4eb1cc468e  2D  (1.0, 1.0)  ...   

                   27                 28                   29  \
0  0.1186632075183146  384.0881508303159  0.09537657526231705   

                  30                   31                  32  \
0  5198.022599575477  0.03699217961542942  0.5176157407259447   

                  33                     34  label             pat  
0  2.916333030932906  0.0006440005212846345    nan  sample_dcm_roi  

[1 rows x 37 columns]
(512, 512)
(512, 512)


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


...

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
21  v3.0.1

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
21  v3.0.1

         0       1      2      3        4  \
0   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
1   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
2   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
3   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
4   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
5   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
6   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
7   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
8   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
9   v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
10  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
11  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
12  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
13  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
14  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
15  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
16  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
17  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
18  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
19  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
20  v3.0.1  1.25.2  2.3.1  1.6.0  3.10.12   
21  v3.0.1